# Session 3 - Explainable AI and Model Debugging

## Storyline

`Selected legitimate model -> global reliance -> local prediction -> wrong-case autopsy -> stability -> next test`

The notebook is designed so every output can be explained in words. After each major code cell, an **output explanation block** states what the output means and what must not be overclaimed.

## Live workspace rule
Cells with `...` are intentionally incomplete. Replace them only after reading the nearby hint and symbol explanation.


## Symbol glossary used in this notebook

| Symbol | Meaning | How to explain it |
|---|---|---|
| $x$ | One input case | One region-month record supplied to the model |
| $X$ | The complete input dataset | All region-month records used as model inputs |
| $x_j$ | Feature $j$ in case $x$ | One value in one record, such as `drought_index` |
| $j$ | Feature index | The number or position of one feature |
| $M$ | Total number of features | The number of input columns used by the model |
| $c$ | Class index | One class: `LOW`, `MEDIUM`, or `HIGH` |
| $C$ | Total number of classes | The number of output classes; here, $C=3$ |
| $c^*(x)$ | Predicted class for case $x$ | The class with the maximum predicted probability |
| $P_c$ | Precision for class $c$ | Of all cases predicted as class $c$, how many were correct? |
| $R_c$ | Recall for class $c$ | Of all cases truly belonging to class $c$, how many were detected? |
| $TP_c$ | True positives for class $c$ | Cases correctly predicted as class $c$ |
| $FP_c$ | False positives for class $c$ | Cases incorrectly predicted as class $c$ |
| $FN_c$ | False negatives for class $c$ | Cases belonging to class $c$ that the model missed |
| $F_{1,c}$ | F1 score for class $c$ | The balance between precision and recall for one class |
| $F_{1,\mathrm{macro}}$ | Macro F1 score | The average F1 score across all classes |
| $S_0$ | Baseline score | Macro F1 before any feature is shuffled |
| $S^{\mathrm{perm}}_{j,r}$ | Permuted score | Macro F1 after feature $j$ is shuffled in repetition $r$ |
| $r$ | Permutation repetition index | One individual shuffle experiment |
| $R$ | Total permutation repetitions | The number of times each feature is shuffled |
| $I_j$ | Permutation importance of feature $j$ | The average score decrease caused by shuffling feature $j$ |
| $\hat{p}_c(x)$ | Predicted probability of class $c$ | The model's confidence that case $x$ belongs to class $c$ |
| $\hat{p}_{c^*}(x)$ | Probability of the predicted class | The model's confidence in its selected class |
| $b_j$ | Baseline value for feature $j$ | The reference value used during occlusion, such as the training median |
| $x_{j\leftarrow b_j}$ | Modified input case | Case $x$ after feature $j$ is replaced by baseline $b_j$ |
| $\Delta_{j,c^*}$ | Local occlusion effect | The change in predicted-class probability after feature $j$ is replaced |
| $f_c(x)$ | Model output for class $c$ | The class-specific model output being explained |
| $f_c(X)$ | Model outputs for class $c$ over dataset $X$ | Class-$c$ outputs across the reference data |
| $\mathbb{E}[f_c(X)]$ | SHAP baseline | The average or reference model output for class $c$ |
| $\phi_{j,c}$ | SHAP value | The contribution of feature $j$ toward class $c$ |

### Equations and why they matter

#### 1. F1 score for one class

$$
F_{1,c}
=
\frac{2P_cR_c}{P_c+R_c}
=
\frac{2TP_c}{2TP_c+FP_c+FN_c}
$$

where:

- $P_c$ is precision for class $c$.
- $R_c$ is recall for class $c$.
- $TP_c$ is the number of true positives for class $c$.
- $FP_c$ is the number of false positives for class $c$.
- $FN_c$ is the number of false negatives for class $c$.

**Why it matters:** Each class must have both usable precision and usable recall.

---

#### 2. Macro F1 score

$$
F_{1,\mathrm{macro}}
=
\frac{1}{C}
\sum_{c=1}^{C}
F_{1,c}
$$

where:

- $C$ is the total number of classes.
- $c$ identifies one class.
- $F_{1,c}$ is the F1 score for class $c$.

**Why it matters:** Every class receives equal influence, including rare classes such as `HIGH`.

---

#### 3. Permutation importance

$$
I_j
=
\frac{1}{R}
\sum_{r=1}^{R}
\left(
S_0-S^{\mathrm{perm}}_{j,r}
\right)
$$

where:

- $I_j$ is the permutation importance of feature $j$.
- $S_0$ is the original score before shuffling.
- $S^{\mathrm{perm}}_{j,r}$ is the score after feature $j$ is shuffled.
- $r$ identifies one permutation repetition.
- $R$ is the total number of repetitions.

**Why it matters:** This measures how much the fitted model depends on feature $j$ for the selected dataset and evaluation metric.

- A large positive $I_j$ indicates strong model reliance.
- An $I_j$ near zero indicates little measurable reliance.
- A negative $I_j$ means that performance improved after shuffling.

Permutation importance measures model reliance, not real-world causality.

---

#### 4. Local occlusion effect

$$
\Delta_{j,c^*}
=
\hat{p}_{c^*}(x)
-
\hat{p}_{c^*}
\left(
x_{j\leftarrow b_j}
\right)
$$

where:

- $\Delta_{j,c^*}$ is the local effect of feature $j$ on the predicted class.
- $c^*$ is the class selected by the model.
- $\hat{p}_{c^*}(x)$ is the original predicted-class probability.
- $b_j$ is the baseline value for feature $j$.
- $x_{j\leftarrow b_j}$ is the modified case after replacing feature $j$.
- $\hat{p}_{c^*}(x_{j\leftarrow b_j})$ is the probability after replacement.

**Why it matters:** This measures how the current feature value changes the predicted-class probability compared with a training reference.

- $\Delta_{j,c^*}>0$ means that the feature pushes the prediction toward the selected class.
- $\Delta_{j,c^*}<0$ means that the feature pushes the prediction away from the selected class.
- A large absolute value of $\Delta_{j,c^*}$ indicates strong local sensitivity.

---

#### 5. SHAP additive explanation

$$
f_c(x)
=
\mathbb{E}[f_c(X)]
+
\sum_{j=1}^{M}
\phi_{j,c}
$$

where:

- $f_c(x)$ is the model output for class $c$ and case $x$.
- $\mathbb{E}[f_c(X)]$ is the average or reference output for class $c$.
- $\phi_{j,c}$ is the SHAP contribution of feature $j$ toward class $c$.
- $j$ identifies one feature.
- $M$ is the total number of features.

**Why it matters:** The SHAP baseline and all feature contributions reconstruct the model output in the selected output space.

- $\phi_{j,c}>0$ means that feature $j$ pushes the output toward class $c$.
- $\phi_{j,c}<0$ means that feature $j$ pushes the output away from class $c$.
- A larger absolute value of $\phi_{j,c}$ indicates a stronger assigned contribution.

## Interpretation boundaries

- Global importance is not local attribution.
- Positive SHAP means **toward the selected model output**, not a beneficial or causal real-world effect.
- Multiclass SHAP requires a separate vector for LOW, MEDIUM and HIGH.
- SHAP values may be expressed in decision-score or log-odds space rather than direct probability.
- Confidence is descriptive here; calibration is reserved for Session 4.


## 1. Locate the course package


In [ ]:
# Import system utilities so the notebook can locate the complete course package.
import sys

# Import Path for platform-independent filesystem operations.
from pathlib import Path

# Start the search from the current execution directory.
current = Path.cwd().resolve()

# Inspect the current directory and all parent directories.
candidate_roots = [current] + list(current.parents)

# Initialize the resolved project and source directories.
PROJECT_ROOT = None
SOURCE_ROOT = None

# Search for the refactored package first, then the root compatibility shim.
for parent in candidate_roots:
    # Prefer the complete package layout that contains src/course_core.py.
    if (parent / "src" / "course_core.py").exists():
        PROJECT_ROOT = parent
        SOURCE_ROOT = parent / "src"
        break

    # Support execution when the current directory is already src/.
    if parent.name == "src" and (parent / "course_core.py").exists():
        PROJECT_ROOT = parent.parent
        SOURCE_ROOT = parent
        break

    # Support the root-level course_core compatibility shim.
    if (parent / "course_core.py").exists() and (parent / "src").exists():
        PROJECT_ROOT = parent
        SOURCE_ROOT = parent / "src"
        break

# Stop with a precise message rather than failing later with ModuleNotFoundError.
if PROJECT_ROOT is None or SOURCE_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the course package. Keep this notebook inside the extracted "
        "course folder that contains src/course_core.py."
    )

# Insert the source directory before other import locations.
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

# Resolve the existing shared data and report directories.
DATA_DIR = SOURCE_ROOT / "data"
REPORTS_DIR = SOURCE_ROOT / "reports"

# Create the report directory when required.
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Display the resolved paths for auditability.
print(f"Project root: {PROJECT_ROOT}")
print(f"Source root: {SOURCE_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Reports directory: {REPORTS_DIR}")


### How to explain this output
If a path is printed, the notebook has found the course package. If an import error appears later, the root path should be checked first.

## 2. Imports


In [ ]:
# Import NumPy for arrays and numerical operations.
import numpy as np

# Import pandas for tables and artifact export.
import pandas as pd

# Import matplotlib for evidence visualizations.
import matplotlib.pyplot as plt

# Import permutation importance for global model inspection.
from sklearn.inspection import permutation_importance

# Import evaluation metrics for model-behavior evidence.
from sklearn.metrics import confusion_matrix, f1_score

# Import the course implementation and contracts.
from course_core import (
    CLASS_NAMES,
    MODEL_FEATURES,
    RANDOM_STATE,
    encode_target,
    fuse_sources,
    generate_raw_sources,
    train_model_comparison,
)

# Confirm that the environment is ready.
print("Imports resolved.")


### How to explain this output
This confirms that the modeling contracts from Sessions 1 and 2 are available: class names, approved features, and the same training/evaluation functions.

## 3. Load or rebuild the trusted modeling table


In [ ]:
# Resolve raw and processed data directories.
raw_dir = DATA_DIR / "raw"
processed_dir = DATA_DIR / "processed"

# Create the processed directory when it is missing.
processed_dir.mkdir(parents=True, exist_ok=True)

# Define the expected modeling-table path.
modeling_table_path = processed_dir / "modeling_table.csv"

# Generate deterministic sources when raw data is unavailable.
if not raw_dir.exists():
    # Create the controlled raw source files.
    generate_raw_sources(raw_dir)

# Reuse the trusted fused table when available.
if modeling_table_path.exists():
    # Load the fused data from disk.
    data = pd.read_csv(modeling_table_path)
else:
    # Rebuild the fused data and its source audit.
    data, source_audit = fuse_sources(raw_dir, processed_dir)

# Display the resulting dimensions and class distribution.
print(f"Modeling table shape: {data.shape}")
print(data["risk_label"].value_counts().to_string())


### How to explain this output
The row/column count confirms that the fused dataset is available. The class distribution tells whether LOW, MEDIUM, and HIGH are balanced or imbalanced before explanations are calculated.

## 4. Recover the legitimate selected model from Session 2


In [ ]:
# Train the same candidate registry used in Session 2.
comparison = train_model_comparison(data)

# Extract chronological partitions.
train = comparison["splits"]["train"]
calibration = comparison["splits"]["calibration"]
test = comparison["splits"]["test"]

# Copy the common metric table.
metrics = comparison["metrics"].copy()

# Extract the fitted model registry.
models = comparison["models"]

# Declare the legitimate candidate registry before ranking.
eligible_names = ["dummy", "logistic", "advanced_correct"]

# Exclude the deliberately leaky model.
legitimate_metrics = metrics[metrics["model"].isin(eligible_names)].copy()

# Rank legitimate candidates by Macro F1 and then log loss.
legitimate_metrics = legitimate_metrics.sort_values(
    ["macro_f1", "log_loss", "model"],
    ascending=[False, True, True],
).reset_index(drop=True)

# Store the selected legitimate model name.
selected_name = legitimate_metrics.loc[0, "model"]

# Store the selected fitted model and its approved features.
selected_model, selected_features = models[selected_name]

# Encode future-holdout labels.
y_test = encode_target(test["risk_label"])

# Display the full comparison and the selected model.
display(metrics.sort_values("macro_f1", ascending=False))
print(f"Selected product model: {selected_name}")


### How to explain this output
This recreates the Session 2 decision: the leaky model is excluded first, then legitimate models are ranked. The selected model name is the only model that should be explained as the product model.

## 5. Inspect the selected model's future-holdout behavior

The confusion matrix and class-level F1 values establish the error context before explanations are interpreted.


In [ ]:
# Predict future-holdout class probabilities.
holdout_probabilities = selected_model.predict_proba(test[selected_features])

# FILL: Select the largest-probability class for every case.
y_pred = ...

# FILL: Calculate one F1 value per class.
class_f1 = ...

# FILL: Calculate the unweighted mean of class-level F1 values.
macro_f1 = ...

# HINT: Use f1_score with average=None for class values and average="macro" for Macro F1.
print("Replace the ellipses, then display the class and Macro F1 evidence.")


### How to explain this output
Read the output as evidence for the current step in the investigation. State what the output proves, what it does not prove, and how it moves the casebook forward.

## 6. Global behavior - permutation importance

$$
I_j=s(f,D)-\frac{1}{K}\sum_{k=1}^{K}s\left(f,\widetilde D_{k,j}\right)
$$

**Interpretation:** a large positive value means that disrupting feature $j$ reduced held-out Macro F1. The result is conditional on this model, dataset and metric.


In [ ]:
# Compute repeated permutation importance using the class-balanced metric.
permutation_result = permutation_importance(
    selected_model,
    test[selected_features],
    y_test,
    scoring="f1_macro",
    n_repeats=20,
    random_state=RANDOM_STATE,
    n_jobs=1,
)

# Convert the result into an interpretable table.
permutation_table = pd.DataFrame(
    {
        "feature": selected_features,
        "importance_mean": permutation_result.importances_mean,
        "importance_std": permutation_result.importances_std,
    }
).sort_values("importance_mean", ascending=False).reset_index(drop=True)

# Export the global explanation artifact.
permutation_table.to_csv(
    REPORTS_DIR / "session3_selected_model_permutation_importance.csv",
    index=False,
)

# Display the strongest global dependencies.
display(permutation_table.head(10))


### How to explain this output
A large positive importance means that shuffling the feature caused Macro F1 to drop. The model relied on that signal on this holdout set. A small or negative value should be described as weak, unstable, or redundant - not physically harmful.

In [ ]:
# FILL: Compute permutation importance for the selected product model.
permutation_result = permutation_importance(
    selected_model,
    test[selected_features],
    y_test,
    scoring=...,       # HINT: use the same class-balanced metric from Session 2.
    n_repeats=...,     # HINT: repeated shuffles reveal variability.
    random_state=RANDOM_STATE,
    n_jobs=1,
)

# FILL: Build a dataframe containing feature, mean importance and standard deviation.
permutation_table = ...

# HINT: Read importances_mean and importances_std from permutation_result.


### How to explain this output
A large positive importance means that shuffling the feature caused Macro F1 to drop. The model relied on that signal on this holdout set. A small or negative value should be described as weak, unstable, or redundant - not physically harmful.

## 7. Local occlusion explanation

$$
\Delta_{j,c}(x)=p_c(x)-p_c\left(x_{j\leftarrow r_j}\right)
$$

A positive effect means the original feature value increased support for class $c$ relative to the reference. A negative effect means it reduced support.


In [ ]:
# Define a model-agnostic local probability-sensitivity function.
def local_occlusion_table(model, sample, reference, features, class_id):
    # Calculate the original selected-class probability.
    base_probability = model.predict_proba(sample[features])[0, class_id]

    # Create a collection for feature-level effects.
    records = []

    # Replace one feature at a time.
    for feature in features:
        # Copy the case so the original row remains unchanged.
        altered = sample.copy()

        # Replace the selected feature with its reference value.
        altered.loc[:, feature] = reference[feature]

        # Recalculate the selected-class probability.
        changed_probability = model.predict_proba(altered[features])[0, class_id]

        # Measure the support contributed by the original feature value.
        effect = base_probability - changed_probability

        # Store one interpretable record.
        records.append(
            {
                "feature": feature,
                "case_value": float(sample.iloc[0][feature]),
                "reference_value": float(reference[feature]),
                "base_probability": float(base_probability),
                "changed_probability": float(changed_probability),
                "local_effect": float(effect),
                "absolute_effect": float(abs(effect)),
            }
        )

    # Rank factors by local effect magnitude.
    return pd.DataFrame(records).sort_values(
        "absolute_effect",
        ascending=False,
    ).reset_index(drop=True)

# Use development medians as the primary reference profile.
reference_medians = train[selected_features].median()

# Confirm that the explainer is ready.
print("Local occlusion function ready.")


### How to explain this output
This function creates a local sensitivity explanation. It asks: if only this feature were replaced by a typical training value, how much would the selected-class probability change?

## 8. Build one representative local case for every predicted class


In [ ]:
# Define a model-agnostic local probability-sensitivity function.
def local_occlusion_table(model, sample, reference, features, class_id):
    # FILL: Calculate the original selected-class probability.
    base_probability = ...

    # Create a collection for feature-level effects.
    records = []

    # Replace one feature at a time.
    for feature in features:
        # Copy the case so the original row remains unchanged.
        altered = sample.copy()

        # FILL: Replace the selected feature with its reference value.
        altered.loc[:, feature] = ...

        # FILL: Recalculate the selected-class probability.
        changed_probability = ...

        # FILL: Apply Delta = original probability - changed probability.
        effect = ...

        # Store one interpretable record.
        records.append(
            {
                "feature": feature,
                "case_value": float(sample.iloc[0][feature]),
                "reference_value": float(reference[feature]),
                "base_probability": float(base_probability),
                "changed_probability": float(changed_probability),
                "local_effect": float(effect),
                "absolute_effect": float(abs(effect)),
            }
        )

    # Return the factors ordered by effect magnitude.
    return pd.DataFrame(records).sort_values(
        "absolute_effect",
        ascending=False,
    ).reset_index(drop=True)

# Use development medians as the reference profile.
reference_medians = train[selected_features].median()


### How to explain this output
This function creates a local sensitivity explanation. It asks: if only this feature were replaced by a typical training value, how much would the selected-class probability change?

## 9. Apply class-aware SHAP to the selected logistic model

$$
g_c(f(x))=\phi_{0,c}+\sum_{j=1}^{M}\phi_{j,c}
$$

The selected model is a pipeline. Its fitted preprocessing stage must transform both the background data and explained cases before `LinearExplainer` is applied to the fitted logistic classifier.


In [ ]:
# Import SHAP for additive feature attribution.
import shap

# Extract the fitted preprocessing stage from the selected pipeline.
preprocessor = selected_model.named_steps["prepare"]

# Extract the fitted logistic classifier from the selected pipeline.
linear_model = selected_model.named_steps["model"]

# Transform development data with the fitted preprocessing contract.
X_background = preprocessor.transform(train[selected_features])

# Transform future-holdout data with the same fitted contract.
X_explain = preprocessor.transform(test[selected_features])

# Convert sparse matrices when required by the explainer.
if hasattr(X_background, "toarray"):
    X_background = X_background.toarray()

# Convert sparse holdout matrices when required.
if hasattr(X_explain, "toarray"):
    X_explain = X_explain.toarray()

# Retrieve readable transformed feature names.
transformed_feature_names = [
    name.split("__", 1)[-1]
    for name in preprocessor.get_feature_names_out()
]

# Create a compact representative background for live execution.
linear_explainer = shap.LinearExplainer(
    linear_model,
    X_background[:120],
)

# Explain all future-holdout cases in the classifier decision-score space.
linear_explanation = linear_explainer(X_explain)

# Display the multiclass tensor shape.
print("Linear SHAP shape:", np.asarray(linear_explanation.values).shape)


### How to explain this output
The selected logistic model is explained directly. The output shape tells whether SHAP returned one attribution vector or one vector per class. For multiclass models, the class axis must be handled carefully.

In [ ]:
# Define a helper that extracts one case and one class from SHAP outputs.
def class_shap_table(explanation, case_index, class_id, feature_names, transformed_values):
    # Convert SHAP values to a NumPy array.
    values = np.asarray(explanation.values)

    # Handle multiclass output: cases x features x classes.
    if values.ndim == 3:
        # Select the class-specific feature vector.
        local_values = values[case_index, :, class_id]
    elif values.ndim == 2:
        # Use the direct feature vector for a single-output explainer.
        local_values = values[case_index, :]
    else:
        # Reject an unsupported output structure clearly.
        raise ValueError(f"Unsupported SHAP value shape: {values.shape}")

    # Build an interpretable attribution table.
    table = pd.DataFrame(
        {
            "feature": feature_names,
            "transformed_value": transformed_values[case_index],
            "shap_value": local_values,
            "absolute_shap": np.abs(local_values),
        }
    )

    # Rank by contribution magnitude.
    return table.sort_values("absolute_shap", ascending=False).reset_index(drop=True)

# Confirm that the extraction helper is ready.
print("Class-aware SHAP extraction ready.")


### How to explain this output
This helper extracts the correct class-specific SHAP vector. Positive values move the selected class score upward; negative values move it downward.

In [ ]:
# Import SHAP for additive feature attribution.
import shap

# FILL: Extract the fitted preprocessing stage from the selected pipeline.
preprocessor = ...

# FILL: Extract the fitted logistic classifier from the selected pipeline.
linear_model = ...

# FILL: Transform development and holdout data with the fitted preprocessor.
X_background = ...
X_explain = ...

# Convert sparse matrices when required.
if hasattr(X_background, "toarray"):
    X_background = X_background.toarray()
if hasattr(X_explain, "toarray"):
    X_explain = X_explain.toarray()

# HINT: The pipeline step names are "prepare" and "model".
# HINT: Use shap.LinearExplainer(linear_model, X_background[:120]).


### How to explain this output
The selected logistic model is explained directly. The output shape tells whether SHAP returned one attribution vector or one vector per class. For multiclass models, the class axis must be handled carefully.

In [ ]:
# Define a parse-safe class-aware SHAP extraction helper.
def class_shap_table(explanation, case_index, class_id, feature_names, transformed_values):
    # Convert SHAP values to a NumPy array.
    values = np.asarray(explanation.values)

    # FILL: For a 3D tensor, select [case, feature, class].
    if values.ndim == 3:
        local_values = ...
    elif values.ndim == 2:
        local_values = values[case_index, :]
    else:
        raise ValueError(f"Unsupported SHAP value shape: {values.shape}")

    # Return a table ordered by absolute contribution.
    table = pd.DataFrame(
        {
            "feature": feature_names,
            "transformed_value": transformed_values[case_index],
            "shap_value": local_values,
            "absolute_shap": np.abs(local_values),
        }
    )
    return table.sort_values("absolute_shap", ascending=False).reset_index(drop=True)

# HINT: Multiclass SHAP commonly has shape (cases, features, classes).


### How to explain this output
This helper extracts the correct class-specific SHAP vector. Positive values move the selected class score upward; negative values move it downward.

## 11. Verify SHAP additivity

$$
g_c(f(x))=\phi_{0,c}+\sum_j\phi_{j,c}
$$

For the selected logistic model, `LinearExplainer` reconstructs the classifier decision score for the selected class.


In [ ]:
# FILL: Locate positions where predicted and actual classes differ.
wrong_positions = ...

# FILL: Select the first genuine wrong prediction.
error_position = ...

# HINT: np.flatnonzero(condition) returns matching positions.
# HINT: y_pred != y_test identifies errors.


### How to explain this output
The first wrong holdout prediction becomes the debugging target. The goal is not to hide the error; the goal is to understand which signals pushed the model toward the wrong class.

## 12. Measure explanation stability

$$
J(A,B)=\frac{|A\cap B|}{|A\cup B|}
$$

Two reasonable SHAP backgrounds are used. The top-five sets are compared with Jaccard similarity.


In [ ]:
# Create two distinct but reasonable development backgrounds.
background_a = X_background[:120]
background_b = X_background[-120:]

# Fit one explainer for each background.
explainer_a = shap.LinearExplainer(linear_model, background_a)
explainer_b = shap.LinearExplainer(linear_model, background_b)

# Explain only the selected wrong case under both backgrounds.
explanation_a = explainer_a(X_explain[[error_position]])
explanation_b = explainer_b(X_explain[[error_position]])

# Build class-aware tables for both explanations.
table_a = class_shap_table(
    explanation_a,
    0,
    error_predicted_class_id,
    transformed_feature_names,
    X_explain[[error_position]],
)
table_b = class_shap_table(
    explanation_b,
    0,
    error_predicted_class_id,
    transformed_feature_names,
    X_explain[[error_position]],
)

# Convert the strongest five factors into sets.
top_a = set(table_a.head(5)["feature"])
top_b = set(table_b.head(5)["feature"])

# Calculate Jaccard similarity.
jaccard_similarity = len(top_a & top_b) / len(top_a | top_b)

# Display the stability evidence.
print("Background A top-5:", sorted(top_a))
print("Background B top-5:", sorted(top_b))
print(f"Jaccard similarity: {jaccard_similarity:.3f}")


### How to explain this output
The selected logistic model is explained directly. The output shape tells whether SHAP returned one attribution vector or one vector per class. For multiclass models, the class axis must be handled carefully.

In [ ]:
# FILL: Create two top-five feature sets named top_a and top_b.
top_a = ...
top_b = ...

# FILL: Calculate J(A,B) = intersection size / union size.
jaccard_similarity = ...

# HINT: Use set intersection (&) and set union (|).
print("Complete the two sets and Jaccard equation.")


### How to explain this output
Jaccard compares the top factors from two different background samples. High overlap means the top explanation drivers are stable; low overlap means the local explanation is sensitive to the chosen background.

In [ ]:
# Extract the strongest SHAP driver for the wrong case.
strongest_shap_driver = error_shap.iloc[0]["feature"]

# Construct one evidence-linked and falsifiable debugging decision.
hypothesis = pd.DataFrame(
    [
        {
            "observed_failure": (
                f"{CLASS_NAMES[error_actual_class_id]} was predicted as "
                f"{CLASS_NAMES[error_predicted_class_id]}"
            ),
            "evidence": (
                f"The strongest class-aware SHAP driver was {strongest_shap_driver}; "
                "disease-related signals were duplicated across raw and engineered features."
            ),
            "hypothesis": (
                "The selected model may over-count correlated disease information near the "
                "MEDIUM/HIGH boundary."
            ),
            "next_test": (
                "Refit with one redundant disease representation removed or with stronger "
                "regularization while preserving the chronological split and metric contract."
            ),
            "success_condition": (
                "MEDIUM-to-HIGH false positives decrease without a material reduction in "
                "HIGH-class recall or future-holdout Macro F1."
            ),
        }
    ]
)

# Export the final debugging decision.
hypothesis.to_csv(REPORTS_DIR / "session3_debugging_hypothesis.csv", index=False)

# Display the final artifact.
display(hypothesis)


### How to explain this output
The final row converts explanation evidence into a scientific debugging decision: observed failure, supporting evidence, testable hypothesis, next experiment, and evaluation metric.

In [ ]:
# Store the strongest SHAP driver after the wrong-case table has been constructed.
strongest_shap_driver = error_shap.iloc[0]["feature"]

# Create a parse-safe hypothesis template.
hypothesis = pd.DataFrame(
    [
        {
            "observed_failure": (
                f"{CLASS_NAMES[error_actual_class_id]} was predicted as "
                f"{CLASS_NAMES[error_predicted_class_id]}"
            ),
            "evidence": f"The strongest class-aware SHAP driver was {strongest_shap_driver}.",
            "hypothesis": "TODO: write one falsifiable mechanism suggested by the explanation evidence.",
            "next_test": "TODO: change one factor while keeping the split and metric contract frozen.",
            "success_condition": "TODO: state the measurable result that would support the hypothesis.",
        }
    ]
)

# Display the template for completion.
display(hypothesis)


### How to explain this output
The final row converts explanation evidence into a scientific debugging decision: observed failure, supporting evidence, testable hypothesis, next experiment, and evaluation metric.

In [ ]:
# Keep the notebook usable when SHAP or the optional nonlinear backend differs by environment.
try:
    # Retrieve the corrected nonlinear pipeline and its feature contract.
    tree_pipeline, tree_features = models["advanced_correct"]

    # Retrieve the fitted preprocessing stage without assuming one package version.
    tree_preprocessor = (
        tree_pipeline.named_steps.get("prepare")
        or tree_pipeline.named_steps.get("preprocessor")
    )

    # Retrieve the fitted tree estimator without passing the surrounding pipeline to SHAP.
    direct_tree_model = (
        tree_pipeline.named_steps.get("model")
        or tree_pipeline.named_steps.get("classifier")
    )

    # Stop clearly if the expected pipeline stages are unavailable.
    if tree_preprocessor is None or direct_tree_model is None:
        raise KeyError("Could not resolve the nonlinear preprocessing and estimator stages.")

    # Transform the future-holdout sample with the fitted preprocessing contract.
    tree_sample = tree_preprocessor.transform(
        test[tree_features].iloc[:30]
    )

    # Convert sparse matrices when the selected transformer returns sparse output.
    if hasattr(tree_sample, "toarray"):
        tree_sample = tree_sample.toarray()

    # Create the optimized tree-specific explainer from the direct fitted estimator.
    tree_explainer = shap.TreeExplainer(direct_tree_model)

    # Calculate tree-specific SHAP values for a bounded live workload.
    tree_explanation = tree_explainer(tree_sample)

    # Report the returned multiclass attribution shape.
    print("Tree SHAP shape:", np.asarray(tree_explanation.values).shape)
except Exception as exc:
    # Explain clearly why the optional block was skipped in this environment.
    print(f"Optional Tree SHAP demonstration skipped: {exc}")


### How to explain this output
This optional block explains the corrected tree candidate with a tree-specific explainer. It is useful for comparison, but it does not replace the explanation of the selected product model.

## Session result

The selected legitimate model now has:

- a metric-aligned global reliance profile;
- representative local probability explanations;
- class-aware Linear SHAP attributions;
- one genuine wrong-case autopsy;
- an additivity verification;
- a background-stability measurement;
- one falsifiable next experiment.

**Next session:** probability calibration, anomaly detection, and ACCEPT / HUMAN REVIEW / REJECT routing.
